In [ ]:
# Uncomment if needed
# !pip install torch torchvision Pillow numpy matplotlib

In [ ]:
# Imports
import sys
import os
import numpy as np
from PIL import Image, UnidentifiedImageError

import torch
from torch.utils.data import Dataset, random_split
import torchvision.transforms as transforms

In [ ]:
# Auto-detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = "https://github.com/your-username/your-repo.git"
    REPO_DIR = "/content/your-repo"
    
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
else:
    src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
    sys.path.insert(0, src_path)

In [ ]:
from dataset import CelebrityFacesDataset, TransformedSubset

In [ ]:
# Set these 
image_folder = ""
output_file  = ""

assert os.path.exists(image_folder), f"Image folder not found: {image_folder}"

In [ ]:
# Check that image files exist
print("First 10 items:", os.listdir(image_folder)[:10])

In [ ]:
# Set random seed
torch.manual_seed(42)

# Set image import mode ('L' for BW, 'RGB' for color)
mode = "L"

# Load data
dataset = CelebrityFacesDataset(image_folder=image_folder, img_mode=mode, transform=None)

# Transforms
train_transform = transforms.Compose([
    transforms.Resize(110),
    transforms.CenterCrop(96),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize(110),
    transforms.CenterCrop(96),
    transforms.ToTensor()
])

# Train and test sizes
train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size

# Split data
train_subset, test_subset = random_split(dataset, [train_size, test_size])

# Transform data
train_dataset = TransformedSubset(train_subset, train_transform)
test_dataset = TransformedSubset(test_subset, test_transform)

# Convert data to numpy array
train_dataset_np = train_dataset.to_numpy()
test_dataset_np = test_dataset.to_numpy()

# If BW, add a dimension to tensor
if mode == "L":
    train_dataset_np = np.expand_dims(train_dataset_np, axis=1)
    test_dataset_np = np.expand_dims(test_dataset_np, axis=1)

In [ ]:
# Print data shapes
print("Train shape:", train_dataset_np.shape)
print("Test shape:", test_dataset_np.shape)

In [ ]:
# Save compressed data
np.savez(output_file, train=train_dataset_np, test=test_dataset_np)